In [1]:

import pandas as pd
df = pd.read_csv(r'C:\Project\Dataset\Churn_Modelling.csv')
print(df.head())


   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         93826.63       0  
4         790

In [ ]:
import sqlite3


conn = sqlite3.connect('C:/Project/Dataset/bank_churn.db')


df.to_sql('customer_churn', conn, if_exists='replace', index=False)


conn.close()

print("Database created and table 'customer_churn' loaded successfully!")

Database created and table 'customer_churn' loaded successfully!


In [ ]:
import sqlite3
import pandas as pd

def run_query(query):
    conn = sqlite3.connect('C:/Project/sql/bank_churn.db')
    result = pd.read_sql_query(query, conn)
    conn.close()
    return result

In [4]:
run_query("SELECT * FROM customer_churn LIMIT 5;")

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


Question A: How many total customers are in this dataset?

In [5]:
run_query("SELECT COUNT(*) AS total_customers FROM customer_churn;")

,total_customers
0,10000


Question B: What is the overall churn rate?

In [6]:
run_query("""
SELECT 
    Exited, 
    COUNT(*) AS customer_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customer_churn), 2) AS percentage
FROM customer_churn
GROUP BY Exited;
""")

,Exited,customer_count,percentage
0,0,7963,79.63
1,1,2037,20.37


Question C: Do older or younger customers leave more often?

In [7]:
run_query("""
SELECT Exited, AVG(Age) AS average_age 
FROM customer_churn 
GROUP BY Exited;
""")

,Exited,average_age
0,0,37.408389
1,1,44.837997


Question D: Does geography play a role? Which country has the highest churn?

In [8]:
run_query("""
SELECT 
    Geography, 
    COUNT(*) AS total_customers,
    SUM(Exited) AS total_churned,
    ROUND(SUM(Exited) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM customer_churn
GROUP BY Geography;
""")

,Geography,total_customers,total_churned,churn_rate_pct
0,France,5014,810,16.15
1,Germany,2509,814,32.44
2,Spain,2477,413,16.67


Question E: Does having a credit card protect against churning?

In [9]:
run_query("""
SELECT 
    HasCrCard, 
    ROUND(SUM(Exited) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM customer_churn
GROUP BY HasCrCard;
""")

,HasCrCard,churn_rate_pct
0,0,20.81
1,1,20.18


Key Insights:

High Churn Baseline: The bank is experiencing a churn rate of 20.37%, indicating a significant retention challenge that impacts long-term profitability.

Demographic Vulnerability: There is a clear age disparity; customers who exit are older (average age of ~45) compared to those who remain (average age of ~37). Retaining this older, likely more stable, demographic should be a priority.  

Geographic Crisis: Customers in Germany exhibit a disproportionately high churn rate (32.44%) compared to France (16.15%) and Spain (16.67%). This requires immediate investigation into regional competitive factors or operational issues.  

Feature Ineffectiveness: Possession of a credit card shows no significant impact on loyalty, with churn rates being nearly identical between credit card holders (20.18%) and non-holders (20.81%). 

Account Balances & Salaries

In [ ]:

import sqlite3

def run_query(query):
    conn = sqlite3.connect('C:/Project/sql/bank_churn.db')
    cursor = conn.cursor()
    cursor.execute(query)
    
    columns = [description[0] for description in cursor.description]
    rows = cursor.fetchall()
    
    conn.close()
    
    print(" | ".join(columns))
    print("-" * 50)
    for row in rows:
        print(" | ".join(str(val) for val in row))

In [11]:
run_query("""
SELECT 
    Exited, 
    ROUND(AVG(Balance), 2) AS avg_balance,
    ROUND(AVG(EstimatedSalary), 2) AS avg_salary,
    COUNT(CASE WHEN Balance = 0 THEN 1 END) AS zero_balance_count,
    ROUND(COUNT(CASE WHEN Balance = 0 THEN 1 END) * 100.0 / COUNT(*), 2) AS zero_balance_pct
FROM customer_churn
GROUP BY Exited;
""")

Exited | avg_balance | avg_salary | zero_balance_count | zero_balance_pct
--------------------------------------------------
0 | 72745.3 | 99738.39 | 3117 | 39.14
1 | 91108.54 | 101465.68 | 500 | 24.55


Customers who leave actually have MORE money in the bank:
The average balance for churned customers (Exited = 1) is ~$91,108, compared to ~$72,745 for customers who stayed. This confirms the bank isn't just losing active accounts, it is losing high-value deposits.

Salary is almost identical:
Estimated salary (~$100k for both groups) shows that income isn't the driving factor here.

Zero-balance accounts aren't the main churn risk:
Nearly 39% of loyal customers have a $0 balance, whereas only 24.5% of churned customers had a $0 balance.

Product Holdings Anomaly

In [12]:
run_query("""
SELECT 
    NumOfProducts, 
    COUNT(*) AS total_customers,
    SUM(Exited) AS churned_customers,
    ROUND(SUM(Exited) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM customer_churn
GROUP BY NumOfProducts
ORDER BY NumOfProducts;
""")

NumOfProducts | total_customers | churned_customers | churn_rate_pct
--------------------------------------------------
1 | 5084 | 1409 | 27.71
2 | 4590 | 348 | 7.58
3 | 266 | 220 | 82.71
4 | 60 | 60 | 100.0


2 Products is the Sweet Spot: Customers with 2 products (like a checking account + savings account) have a tiny churn rate of only 7.58%. They are by far the bank's most loyal group!

1 Product is Vulnerable: Customers with just 1 product churn at 27.71%.

3 or 4 Products is a Total Disaster:

Customers with 3 products churn at a massive 82.71%.

Customers with 4 products churn at 100% (every single one of them left!).

Member Engagement

In [13]:
run_query("""
SELECT 
    IsActiveMember, 
    COUNT(*) AS total_customers,
    SUM(Exited) AS churned_customers,
    ROUND(SUM(Exited) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM customer_churn
GROUP BY IsActiveMember;
""")

IsActiveMember | total_customers | churned_customers | churn_rate_pct
--------------------------------------------------
0 | 4849 | 1302 | 26.85
1 | 5151 | 735 | 14.27


Inactive members (0) churn at almost double the rate (26.85%) of active members (14.27%)!

**Key Insights**


High-Value Accounts Leaving: Churned customers hold significantly higher average balances ($91,108) compared to retained customers ($72,745). The bank is losing high-deposit clients, not dead weight.

Product Saturation Risk:

Customers with 2 products are the most loyal, with a churn rate of only 7.58%.

Customers with 3 products churn at 82.71%.

Customers with 4 products have a 100% churn rate. Aggressive product bundling is actively pushing customers away.

Engagement Protects Loyalty: Inactive members churn at 26.85%, whereas active members churn at 14.27%.

**Strategic Takeaways**

Target 1-Product Customers: Cross-sell a second product to 1-product customers to lock in their loyalty (moving them into the 7.58% churn tier).

Halt 3+ Product Bundling: Investigate multi-product packages immediately to understand why customers with 3 or 4 products universally leave.

Re-engagement Campaigns: Implement targeted email/app push campaigns for inactive members to boost participation and cut churn in half.

**Check for Missing Values & Data Integrity**

In [14]:
run_query("""
SELECT 
    SUM(CASE WHEN CustomerId IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
    SUM(CASE WHEN Age IS NULL THEN 1 ELSE 0 END) AS null_age,
    SUM(CASE WHEN Balance IS NULL THEN 1 ELSE 0 END) AS null_balance,
    SUM(CASE WHEN CreditScore IS NULL THEN 1 ELSE 0 END) AS null_credit_score,
    SUM(CASE WHEN Geography IS NULL THEN 1 ELSE 0 END) AS null_geography
FROM customer_churn;
""")

null_customer_id | null_age | null_balance | null_credit_score | null_geography
--------------------------------------------------
0 | 0 | 0 | 0 | 0


hence dataset is clean.

Feature Engineering

Grouping with age and balance to make it easier to visualize and understand.

In [15]:
run_query("""
CREATE VIEW IF NOT EXISTS vw_churn_analysis AS
SELECT 
    *,
    CASE 
        WHEN Age < 30 THEN 'Under 30'
        WHEN Age BETWEEN 30 AND 39 THEN '30-39'
        WHEN Age BETWEEN 40 AND 49 THEN '40-49'
        WHEN Age BETWEEN 50 AND 59 THEN '50-59'
        ELSE '60+' 
    END AS Age_Group,
    
    CASE 
        WHEN Balance = 0 THEN 'Zero Balance'
        WHEN Balance < 50000 THEN 'Low Balance (<50k)'
        WHEN Balance BETWEEN 50000 AND 120000 THEN 'Medium Balance (50k-120k)'
        ELSE 'High Balance (>120k)'
    END AS Balance_Tier
FROM customer_churn;
""")

print("View created successfully!")

TypeError: 'NoneType' object is not iterable

In [ ]:
import sqlite3

def run_query(query):
    conn = sqlite3.connect('C:/Project/sql/bank_churn.db')
    cursor = conn.cursor()
    cursor.execute(query)
    
    if cursor.description:
        columns = [description[0] for description in cursor.description]
        rows = cursor.fetchall()
        conn.close()
        
        print(" | ".join(columns))
        print("-" * 50)
        for row in rows:
            print(" | ".join(str(val) for val in row))
    else:
        conn.commit()
        conn.close()
        print("Query executed successfully (no rows returned).")

In [ ]:
run_query("""
CREATE VIEW IF NOT EXISTS vw_churn_analysis AS
SELECT 
    *,
    CASE 
        WHEN Age < 30 THEN 'Under 30'
        WHEN Age BETWEEN 30 AND 39 THEN '30-39'
        WHEN Age BETWEEN 40 AND 49 THEN '40-49'
        WHEN Age BETWEEN 50 AND 59 THEN '50-59'
        ELSE '60+' 
    END AS Age_Group,
    
    CASE 
        WHEN Balance = 0 THEN 'Zero Balance'
        WHEN Balance < 50000 THEN 'Low Balance (<50k)'
        WHEN Balance BETWEEN 50000 AND 120000 THEN 'Medium Balance (50k-120k)'
        ELSE 'High Balance (>120k)'
    END AS Balance_Tier
FROM customer_churn;
""")

print("View created successfully!")

Query executed successfully (no rows returned).
View created successfully!


Verifying the New View

In [ ]:
run_query("""
SELECT 
    Age_Group, 
    COUNT(*) AS total_customers,
    SUM(Exited) AS churned_customers,
    ROUND(SUM(Exited) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM vw_churn_analysis
GROUP BY Age_Group
ORDER BY churn_rate_pct DESC;
""")

Age_Group | total_customers | churned_customers | churn_rate_pct
--------------------------------------------------
50-59 | 869 | 487 | 56.04
40-49 | 2618 | 806 | 30.79
60+ | 526 | 147 | 27.95
30-39 | 4346 | 473 | 10.88
Under 30 | 1641 | 124 | 7.56


50–59 is the Danger Zone: Over 56% of customers aged 50 to 59 are leaving! That is more than half of that entire demographic walking out the door.

40–49 is right behind it: Sitting at a 30.79% churn rate.

Under 40 is remarkably loyal:

30–39: Only 10.88% churn.

Under 30: A tiny 7.56% churn rate.

Testing balance tiers:

In [ ]:
run_query("""
SELECT 
    Balance_Tier, 
    COUNT(*) AS total_customers,
    SUM(Exited) AS churned_customers,
    ROUND(SUM(Exited) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM vw_churn_analysis
GROUP BY Balance_Tier
ORDER BY churn_rate_pct DESC;
""")

Balance_Tier | total_customers | churned_customers | churn_rate_pct
--------------------------------------------------
Low Balance (<50k) | 75 | 26 | 34.67
High Balance (>120k) | 3181 | 765 | 24.05
Medium Balance (50k-120k) | 3127 | 746 | 23.86
Zero Balance | 3617 | 500 | 13.82


Zero Balance accounts have only a 13.82% churn rate, while accounts with money in them (Medium, High, and Low Balances) churn between 23.86% and 34.67%.

This confirms our biggest finding of the day: The bank is not losing inactive or empty accounts; it is actively losing its funded, revenue-generating customers.

Exporting Cleaned Data

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('C:/Project/sql/bank_churn.db')

# 1. Export the full view with engineered features
df_view = pd.read_sql_query("SELECT * FROM vw_churn_analysis;", conn)
df_view.to_csv('C:/Project/Dataset/churn_analysis_cleaned.csv', index=False)

df_age = pd.read_sql_query("""
    SELECT Age_Group, COUNT(*) AS total_customers, SUM(Exited) AS churned_customers,
           ROUND(SUM(Exited) * 100.0 / COUNT(*), 2) AS churn_rate_pct
    FROM vw_churn_analysis GROUP BY Age_Group;
""", conn)
df_age.to_csv('C:/Project/Dataset/summary_age_group.csv', index=False)

df_geo = pd.read_sql_query("""
    SELECT Geography, COUNT(*) AS total_customers, SUM(Exited) AS churned_customers,
           ROUND(SUM(Exited) * 100.0 / COUNT(*), 2) AS churn_rate_pct
    FROM vw_churn_analysis GROUP BY Geography;
""", conn)
df_geo.to_csv('C:/Project/Dataset/summary_geography.csv', index=False)

conn.close()
print("All cleaned CSV files exported successfully!") 

All cleaned CSV files exported successfully!
